Import libraries ↓

In [49]:
import pandas as pd
import numpy as np
import pickle
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report
from sklearn.utils.class_weight import compute_class_weight

Data laden ↓

In [50]:
df = pd.read_csv('trainingsdata.csv')

FEATURES = [
    'speler_x_midden', 'speler_x_links', 'ruimte_rechts',
    'steen_x', 'steen_y', 'steen_snelheid',
    'relatief_verschil', 'aantal_stenen', 'score'
]

X = df[FEATURES].values
y = df['actie'].values + 1

print(f'Rijen geladen: {len(df)}')

Rijen geladen: 108959


Normaliseren en splitsen ↓

In [51]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.15, random_state=42
)

print(f'Trainingsset: {len(X_train)} rijen')
print(f'Testset:      {len(X_test)} rijen')

Trainingsset: 92615 rijen
Testset:      16344 rijen


Model bouwen ↓

In [52]:
model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(len(FEATURES),)),
    tf.keras.layers.Dense(128, activation="relu"),
    tf.keras.layers.Dense(128, activation="relu"),
    tf.keras.layers.Dense(64,  activation="relu"),
    tf.keras.layers.Dense(64,  activation="relu"),
    tf.keras.layers.Dense(3,   activation="softmax")
])

model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

Model trainen ↓

In [53]:
klassen = np.unique(y_train)
gewichten = compute_class_weight('balanced', classes=klassen, y=y_train)
class_weight_dict = dict(zip(klassen, gewichten))

model.fit(X_train, y_train, epochs=20, batch_size=64, validation_split=0.1, class_weight=class_weight_dict)

Epoch 1/20
1303/1303 ━━━━━━━━━━━━━━━━━━━━ 11s 5ms/step - accuracy: 0.5950 - loss: 0.7493 - val_accuracy: 0.6284 - val_loss: 0.8179
Epoch 2/20
1303/1303 ━━━━━━━━━━━━━━━━━━━━ 6s 5ms/step - accuracy: 0.6212 - loss: 0.6815 - val_accuracy: 0.5886 - val_loss: 0.8761
Epoch 3/20
1303/1303 ━━━━━━━━━━━━━━━━━━━━ 6s 5ms/step - accuracy: 0.6297 - loss: 0.6699 - val_accuracy: 0.6291 - val_loss: 0.7840
Epoch 4/20
1303/1303 ━━━━━━━━━━━━━━━━━━━━ 7s 5ms/step - accuracy: 0.6239 - loss: 0.6669 - val_accuracy: 0.5974 - val_loss: 0.8514
Epoch 5/20
1303/1303 ━━━━━━━━━━━━━━━━━━━━ 7s 5ms/step - accuracy: 0.6274 - loss: 0.6608 - val_accuracy: 0.5670 - val_loss: 0.9191
Epoch 6/20
1303/1303 ━━━━━━━━━━━━━━━━━━━━ 8s 6ms/step - accuracy: 0.6209 - loss: 0.6561 - val_accuracy: 0.6313 - val_loss: 0.8174
Epoch 7/20
1303/1303 ━━━━━━━━━━━━━━━━━━━━ 7s 5ms/step - accuracy: 0.6244 - loss: 0.6519 - val_accuracy: 0.6444 - val_loss: 0.7886
Epoch 8/20
1303/1303 ━━━━━━━━━━━━━━━━━━━━ 8s 6ms/step - accuracy: 0.6163 - loss: 0.6491 -

Evaluatie ↓

In [54]:
loss, accuracy = model.evaluate(X_test, y_test)
print(f'Accuracy: {accuracy * 100:.1f}%')
print(f'Loss:     {loss:.4f}')
print()

y_pred = np.argmax(model.predict(X_test), axis=1)
print(classification_report(
    y_test, y_pred,
    target_names=['Links', 'Stil', 'Rechts'],
    zero_division=0
))

511/511 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 0.6191 - loss: 0.7891
Accuracy: 61.9%
Loss:     0.7891

511/511 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step
              precision    recall  f1-score   support

       Links       0.22      0.79      0.35      1131
        Stil       0.96      0.59      0.73     14074
      Rechts       0.24      0.77      0.37      1139

    accuracy                           0.62     16344
   macro avg       0.47      0.72      0.48     16344
weighted avg       0.86      0.62      0.68     16344



Model en scaler opslaan ↓

In [55]:
model.save('model_keras.keras')

with open('scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)

print('Model opgeslagen als model_keras.keras')
print('Scaler opgeslagen als scaler.pkl')
print('Klaar! Voer nu ai_speler.py uit.')

Model opgeslagen als model_keras.keras
Scaler opgeslagen als scaler.pkl
Klaar! Voer nu ai_speler.py uit.


Testcase opslaan in Excel ↓

In [56]:
from sklearn.metrics import precision_score, recall_score
import openpyxl
import os
from openpyxl import load_workbook
from datetime import datetime

EXCEL_BESTAND = 'testcases.xlsx'

# Lagen en neuronen ophalen uit het model
lagen_info = [f"{laag.units}" for laag in model.layers if hasattr(laag, 'units')]
lagen_str = ' → '.join(lagen_info)

# Precision en recall per klasse
precision_links = precision_score(y_test, y_pred, labels=[0], average='macro', zero_division=0)
precision_stil = precision_score(y_test, y_pred, labels=[1], average='macro', zero_division=0)
precision_rechts = precision_score(y_test, y_pred, labels=[2], average='macro', zero_division=0)

recall_links = recall_score(y_test, y_pred, labels=[0], average='macro', zero_division=0)
recall_stil = recall_score(y_test, y_pred, labels=[1], average='macro', zero_division=0)
recall_rechts = recall_score(y_test, y_pred, labels=[2], average='macro', zero_division=0)

nieuwe_rij = [
    datetime.now().strftime('%Y-%m-%d %H:%M'),  # datum en tijd
    len(df),  # aantal rijen in dataset
    0.15,  # test size
    len(lagen_info),  # aantal lagen
    lagen_str,  # neuronen per laag
    20,  # epochs
    64,  # batch size
    round(accuracy * 100, 1),  # accuracy %
    round(precision_links * 100, 1),  # precision links %
    round(precision_stil * 100, 1),  # precision stil %
    round(precision_rechts * 100, 1),  # precision rechts %
    round(recall_links * 100, 1),  # recall links %
    round(recall_stil * 100, 1),  # recall stil %
    round(recall_rechts * 100, 1),  # recall rechts %
]

HEADER = [
    'Datum/tijd', 'Rijen dataset', 'Test size', 'Aantal lagen',
    'Neuronen per laag', 'Epochs', 'Batch size',
    'Accuracy %',
    'Precision Links %', 'Precision Stil %', 'Precision Rechts %',
    'Recall Links %', 'Recall Stil %', 'Recall Rechts %',
]

# Excel aanmaken of openen en rij toevoegen
if os.path.exists(EXCEL_BESTAND):
    wb = load_workbook(EXCEL_BESTAND)
    ws = wb.active
else:
    wb = openpyxl.Workbook()
    ws = wb.active
    ws.title = 'Testcases'
    ws.append(HEADER)

ws.append(nieuwe_rij)
wb.save(EXCEL_BESTAND)
print(f'Testcase opgeslagen in {EXCEL_BESTAND}')
print(f'Excel opgeslagen op: {os.path.abspath(EXCEL_BESTAND)}')

Testcase opgeslagen in testcases.xlsx
Excel opgeslagen op: C:\Users\tariq\PycharmProjects\NeuraleNetwerken\Game\testcases.xlsx
